In [44]:
import numpy as np
import pandas as pd
import os
import shutil
import matplotlib.pyplot as plt
import datetime
import cv2 as cv
from sklearn.metrics import r2_score
from scipy.stats import pearsonr,spearmanr
import matplotlib.dates as mdates
import datetime
from sklearn.decomposition import PCA, KernelPCA
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, GridSearchCV
import joblib as jb
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, AdaBoostRegressor, BaggingRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.cluster import SpectralClustering, MiniBatchKMeans, SpectralBiclustering, SpectralCoclustering
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import Ridge, Lasso
import cv2 as cv
from skimage.metrics import structural_similarity
import matplotlib.dates as mdates



In [45]:
#INPUT CSV DATA
belur=pd.read_csv('belur.csv')
belur_frb=pd.read_csv('belur_frb.csv')

In [46]:
#EXTRACTING BELUR DATA
belur['Day'] = pd.to_numeric(belur['ImageFile'].str[9:11])
belur['Month'] = belur['ImageFile'].str[5:8]
belur['Hour'] =pd.to_numeric(belur['ImageFile'].str[12:14])
belur['Min'] = pd.to_numeric(belur['ImageFile'].str[15:17])

In [47]:
#EXTRACTING BELUR_RBU DATA
belur_frb['Day'] = pd.to_numeric(belur_frb['ImageFile'].str[9:11])
belur_frb['Month'] = belur_frb['ImageFile'].str[5:8]
belur_frb['Hour'] =pd.to_numeric(belur_frb['ImageFile'].str[12:14])
belur_frb['Min'] = pd.to_numeric(belur_frb['ImageFile'].str[15:17])

In [48]:
#TURNING MONTHS INTO NUMBERS
Months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

m =[]
for i,r in belur.iterrows():
    s = str(r['Month'])
    i = Months.index(s)
    m.append(i+1)

belur['Month_num']=m

In [49]:
#CONVERTING BELUR DAY,MONTH,HOUR,MIN INTO CYCLIC DATA
belur['Day_sin'] = np.sin(2 * np.pi * belur['Day']/31)
belur['Day_cos'] = np.cos(2 * np.pi * belur['Day']/31)
belur['Mon_sin'] = np.sin(2 * np.pi * belur['Month_num']/12)
belur['Mon_cos'] = np.cos(2 * np.pi * belur['Month_num']/12)
belur['Hour_sin'] = np.sin(2 * np.pi * belur['Hour']/24)
belur['Hour_cos'] = np.cos(2 * np.pi * belur['Hour']/24)
belur['Min_sin'] = np.sin(2 * np.pi * belur['Min']/60)
belur['Min_cos'] = np.cos(2 * np.pi * belur['Min']/60)

In [50]:
m= []
for i,r in belur_frb.iterrows():
    s= str(r["Month"])
    i= Months.index(s)
    m.append(i+1)
belur_frb['Month_num'] = m

In [51]:
#CONVERTING BELUR_RBU DAY,MONTH,HOUR,MIN INTO CYCLIC DATA
belur_frb['Day_sin'] = np.sin(2 * np.pi * belur_frb['Day']/31)
belur_frb['Day_cos'] = np.cos(2 * np.pi * belur_frb['Day']/31)
belur_frb['Mon_sin'] = np.sin(2 * np.pi * belur_frb['Month_num']/12)
belur_frb['Mon_cos'] = np.cos(2 * np.pi * belur_frb['Month_num']/12)
belur_frb['Hour_sin'] = np.sin(2 * np.pi * belur_frb['Hour']/24)
belur_frb['Hour_cos'] = np.cos(2 * np.pi * belur_frb['Hour']/24)
belur_frb['Min_sin'] = np.sin(2 * np.pi * belur_frb['Min']/60)
belur_frb['Min_cos'] = np.cos(2 * np.pi * belur_frb['Min']/60)

In [52]:
#SPLIT BELUR DATA
train_b,test_b = train_test_split(belur,test_size=0.2,random_state=42)

In [53]:
#SPLIT BELUR_RBU DATA
train_bf,test_bf = train_test_split(belur_frb,test_size=0.2,random_state=42)

In [54]:
#NORMALIZING BELUR PM2.5 VALUES
train_b['PM2.5_Norm'] = (train_b['PM2.5'] - train_b['PM2.5'].min())/(train_b['PM2.5'].max() - train_b['PM2.5'].min())
test_b['PM2.5_Norm'] = (test_b['PM2.5'] - train_b['PM2.5'].min())/(train_b['PM2.5'].max() - train_b['PM2.5'].min())

In [55]:
#NORMALIZING BELUR_RBU PM2.5 VALUES
train_bf['PM2.5_Norm'] = (train_bf['PM2.5'] - train_bf['PM2.5'].min())/(train_bf['PM2.5'].max() - train_bf['PM2.5'].min())
test_bf['PM2.5_Norm'] = (test_bf['PM2.5'] - train_bf['PM2.5'].min())/(train_bf['PM2.5'].max() - train_bf['PM2.5'].min())

In [56]:
#CREATING X1 LIST FOR BELUR DATA
X1_train = train_b[['SSIM_ROI','ERODE_SSIM_ROI','Day_sin','Day_cos','Mon_sin','Mon_cos','Hour_sin','Hour_cos']]
X1_test = test_b[['SSIM_ROI','ERODE_SSIM_ROI','Day_sin','Day_cos','Mon_sin','Mon_cos','Hour_sin','Hour_cos']] 


In [57]:
#CREATING X2 LIST FOR BELUR_RBU DATA
X2_train = train_bf[['SSIM_ROI','ERODE_SSIM_ROI','Day_sin','Day_cos','Mon_sin','Mon_cos','Hour_sin','Hour_cos']]
X2_test = test_bf[['SSIM_ROI','ERODE_SSIM_ROI','Day_sin','Day_cos','Mon_sin','Mon_cos','Hour_sin','Hour_cos']]

In [58]:
#CREATING Y1 LIST FOR BELUR DATA
Y1_train= train_b['PM2.5_Norm']
Y1_test= test_b['PM2.5_Norm']

In [59]:
#CREATING Y2 LIST FOR BELUR_RBU DATA
Y2_train= train_bf['PM2.5_Norm']
Y2_test= test_bf['PM2.5_Norm']


In [60]:
#CREATING HYPER_PARAMETER GRID
param_grid = {
    'n_estimators': np.arange(10,1020,10),
    'max_features': ['sqrt','log2',0.1,0.2,0.3,0.5,0.7,1.0],
    'bootstrap': [True,False]
}

In [61]:
#GRID SEARCH FOR BELUR DATA
#CREATE GRID SEARCH
grid = GridSearchCV(ExtraTreesRegressor(),param_grid=param_grid,refit=True,verbose=3,n_jobs=-1)

In [62]:

#TRAIN GRID SEARCH
grid.fit(X1_train,Y1_train)

Fitting 5 folds for each of 1616 candidates, totalling 8080 fits
[CV 1/5] END bootstrap=True, max_features=sqrt, n_estimators=20;, score=0.885 total time=   0.0s
[CV 2/5] END bootstrap=True, max_features=sqrt, n_estimators=30;, score=0.872 total time=   0.0s
[CV 5/5] END bootstrap=True, max_features=sqrt, n_estimators=40;, score=0.833 total time=   0.1s
[CV 1/5] END bootstrap=True, max_features=sqrt, n_estimators=60;, score=0.888 total time=   0.1s
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=70;, score=0.863 total time=   0.1s
[CV 2/5] END bootstrap=True, max_features=sqrt, n_estimators=90;, score=0.878 total time=   0.1s
[CV 1/5] END bootstrap=True, max_features=sqrt, n_estimators=110;, score=0.892 total time=   0.2s
[CV 1/5] END bootstrap=True, max_features=sqrt, n_estimators=130;, score=0.893 total time=   0.2s
[CV 4/5] END bootstrap=True, max_features=sqrt, n_estimators=140;, score=0.882 total time=   0.2s
[CV 2/5] END bootstrap=True, max_features=sqrt, n_estimator

/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV 1/5] END bootstrap=True, max_features=sqrt, n_estimators=10;, score=0.862 total time=   0.0s
[CV 4/5] END bootstrap=True, max_features=sqrt, n_estimators=30;, score=0.868 total time=   0.0s
[CV 2/5] END bootstrap=True, max_features=sqrt, n_estimators=50;, score=0.869 total time=   0.1s
[CV 1/5] END bootstrap=True, max_features=sqrt, n_estimators=70;, score=0.899 total time=   0.1s
[CV 4/5] END bootstrap=True, max_features=sqrt, n_estimators=80;, score=0.887 total time=   0.1s
[CV 2/5] END bootstrap=True, max_features=sqrt, n_estimators=100;, score=0.883 total time=   0.1s
[CV 5/5] END bootstrap=True, max_features=sqrt, n_estimators=110;, score=0.854 total time=   0.1s
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=130;, score=0.881 total time=   0.2s
[CV 5/5] END bootstrap=True, max_features=sqrt, n_estimators=140;, score=0.843 total time=   0.2s
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=160;, score=0.872 total time=   0.2s
[CV 1/5] END bootstrap=Tr

,estimator,ExtraTreesRegressor()
,param_grid,"{'bootstrap': [True, False], 'max_features': ['sqrt', 'log2', ...], 'n_estimators': array([ 10, ... 1000, 1010])}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,np.int64(270)


In [63]:
grid.best_params_

{'bootstrap': True, 'max_features': 1.0, 'n_estimators': np.int64(270)}

In [64]:
grid.score(X1_train,Y1_train)
grid.score(X1_test,Y1_test)

0.9473737889216486

In [22]:
#GRID SEARCH FOR BELUR_RBU DATA
#CREATE GRID SEARCH
grid = GridSearchCV(ExtraTreesRegressor(),param_grid=param_grid,refit=True,verbose=3,n_jobs=-1)

In [23]:

#TRAIN GRID SEARCH
grid.fit(X2_train,Y2_train)

Fitting 5 folds for each of 1616 candidates, totalling 8080 fits

[CV 2/5] END bootstrap=False, max_features=0.5, n_estimators=420;, score=0.927 total time=   0.8s
[CV 5/5] END bootstrap=False, max_features=0.5, n_estimators=430;, score=0.915 total time=   0.8s
[CV 3/5] END bootstrap=False, max_features=0.5, n_estimators=450;, score=0.932 total time=   0.8s
[CV 5/5] END bootstrap=False, max_features=0.5, n_estimators=460;, score=0.914 total time=   0.8s
[CV 3/5] END bootstrap=False, max_features=0.5, n_estimators=480;, score=0.932 total time=   0.8s
[CV 1/5] END bootstrap=False, max_features=0.5, n_estimators=500;, score=0.935 total time=   0.8s
[CV 4/5] END bootstrap=False, max_features=0.5, n_estimators=510;, score=0.933 total time=   0.9s
[CV 2/5] END bootstrap=False, max_features=0.5, n_estimators=530;, score=0.927 total time=   1.0s
[CV 5/5] END bootstrap=False, max_features=0.5, n_estimators=540;, score=0.911 total time=   1.1s
[CV 3/5] END bootstrap=False, max_features=0.5, n_es

/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



[CV 3/5] END bootstrap=True, max_features=1.0, n_estimators=770;, score=0.981 total time=   1.4s
[CV 1/5] END bootstrap=True, max_features=1.0, n_estimators=790;, score=0.966 total time=   1.4s
[CV 3/5] END bootstrap=True, max_features=1.0, n_estimators=800;, score=0.980 total time=   1.4s
[CV 1/5] END bootstrap=True, max_features=1.0, n_estimators=820;, score=0.965 total time=   1.5s
[CV 4/5] END bootstrap=True, max_features=1.0, n_estimators=830;, score=0.971 total time=   1.5s
[CV 2/5] END bootstrap=True, max_features=1.0, n_estimators=850;, score=0.983 total time=   1.5s
[CV 5/5] END bootstrap=True, max_features=1.0, n_estimators=860;, score=0.980 total time=   1.6s
[CV 3/5] END bootstrap=True, max_features=1.0, n_estimators=880;, score=0.979 total time=   1.6s
[CV 1/5] END bootstrap=True, max_features=1.0, n_estimators=900;, score=0.966 total time=   1.6s
[CV 4/5] END bootstrap=True, max_features=1.0, n_estimators=910;, score=0.971 total time=   1.7s
[CV 2/5] END bootstrap=True, 

,estimator,ExtraTreesRegressor()
,param_grid,"{'bootstrap': [True, False], 'max_features': ['sqrt', 'log2', ...], 'n_estimators': array([ 10, ... 1000, 1010])}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,np.int64(160)


In [24]:
grid.best_params_

{'bootstrap': False, 'max_features': 1.0, 'n_estimators': np.int64(160)}

In [26]:
grid.score(X2_train,Y2_train)
grid.score(X2_test,Y2_test)

0.9749162061275138

In [27]:
#BUILD EXTRA TREE REGRESSOR MODELS FOR BELUR DATA
etr_belur = ExtraTreesRegressor(bootstrap = True, max_features = 1.0, n_estimators = 410, random_state=42)
etr_belur.fit(X1_train,Y1_train)


,n_estimators,410
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False



[CV 5/5] END bootstrap=False, max_features=0.5, n_estimators=450;, score=0.979 total time=   0.8s
[CV 3/5] END bootstrap=False, max_features=0.5, n_estimators=470;, score=0.971 total time=   0.8s
[CV 1/5] END bootstrap=False, max_features=0.5, n_estimators=490;, score=0.962 total time=   0.8s
[CV 4/5] END bootstrap=False, max_features=0.5, n_estimators=500;, score=0.973 total time=   0.8s
[CV 2/5] END bootstrap=False, max_features=0.5, n_estimators=520;, score=0.981 total time=   0.9s
[CV 5/5] END bootstrap=False, max_features=0.5, n_estimators=530;, score=0.980 total time=   0.8s
[CV 2/5] END bootstrap=False, max_features=0.5, n_estimators=550;, score=0.981 total time=   0.8s
[CV 4/5] END bootstrap=False, max_features=0.5, n_estimators=560;, score=0.972 total time=   0.9s
[CV 2/5] END bootstrap=False, max_features=0.5, n_estimators=580;, score=0.982 total time=   0.9s
[CV 5/5] END bootstrap=False, max_features=0.5, n_estimators=590;, score=0.979 total time=   1.0s
[CV 3/5] END bootst

In [29]:
etr_belur.score(X1_train,Y1_train)
etr_belur.score(X1_test,Y1_test)

0.9468514486131129

In [31]:
#BUILD EXTRA TREE REGRESSOR MODELS FOR BELUR RBU DATA
etr_belur_frb = ExtraTreesRegressor(bootstrap = False, max_features = 1.0, n_estimators = 160, random_state=42)

In [32]:
etr_belur_frb.fit(X2_train,Y2_train)

,n_estimators,160
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [34]:
etr_belur_frb.score(X2_train,Y2_train)
etr_belur_frb.score(X2_test,Y2_test)

0.9754794346750029

In [35]:
#SAVE THE MODELS AS PICKLE FILES
jb.dump(etr_belur, 'etr_belur.pkl')
jb.dump(etr_belur_frb, 'etr_belur_frb.pkl')

['etr_belur_frb.pkl']

In [36]:
#PREDICT PM2.5 FOR BELUR DATA
predtrain_b = etr_belur.predict(X1_train)
predtest_b = etr_belur.predict(X1_test)

In [37]:
train_b['ETR_ROI'] = predtrain_b * (train_b['PM2.5'].max() - train_b['PM2.5'].min()) + train_b['PM2.5'].min()
test_b['ETR_ROI'] = predtest_b * (train_b['PM2.5'].max() - train_b['PM2.5'].min()) + train_b['PM2.5'].min()

In [38]:
#PREDICT PM2.5 FOR BELUR RBU DATA
predtrain_bf = etr_belur_frb.predict(X2_train)
predtest_bf = etr_belur_frb.predict(X2_test)

In [39]:
train_bf['ETR_ROI'] = predtrain_bf * (train_bf['PM2.5'].max() - train_bf['PM2.5'].min()) + train_bf['PM2.5'].min()
test_bf['ETR_ROI'] = predtest_bf * (train_bf['PM2.5'].max() - train_bf['PM2.5'].min()) + train_bf['PM2.5'].min()


In [40]:
#SAVE AS CSV
#BELUR
train_b.to_csv('belur_train.csv',index=False)
test_b.to_csv('belur_test.csv',index=False)


In [41]:
#BELUR RBU
train_bf.to_csv('belur_frb_train.csv',index=False)
test_bf.to_csv('belur_frb_test.csv',index=False)

In [65]:
#================MODEL II=============================================


[CV 4/5] END bootstrap=False, max_features=0.7, n_estimators=640;, score=0.936 total time=   1.2s
[CV 2/5] END bootstrap=False, max_features=0.7, n_estimators=660;, score=0.932 total time=   1.3s
[CV 5/5] END bootstrap=False, max_features=0.7, n_estimators=670;, score=0.918 total time=   1.2s
[CV 3/5] END bootstrap=False, max_features=0.7, n_estimators=690;, score=0.937 total time=   1.3s
[CV 1/5] END bootstrap=False, max_features=0.7, n_estimators=710;, score=0.937 total time=   1.4s
[CV 4/5] END bootstrap=False, max_features=0.7, n_estimators=720;, score=0.936 total time=   1.4s
[CV 2/5] END bootstrap=False, max_features=0.7, n_estimators=740;, score=0.931 total time=   1.4s
[CV 5/5] END bootstrap=False, max_features=0.7, n_estimators=750;, score=0.919 total time=   1.5s
[CV 3/5] END bootstrap=False, max_features=0.7, n_estimators=770;, score=0.937 total time=   1.4s
[CV 1/5] END bootstrap=False, max_features=0.7, n_estimators=790;, score=0.937 total time=   1.5s
[CV 4/5] END bootst

In [66]:
#MERGE TWO CSV
train_belur_df = pd.merge(train_b,train_bf, on='ImageFile',how='inner')
test_belur_df = pd.merge(test_b,test_bf, on='ImageFile',how='inner')

In [67]:
train_belur_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1531 entries, 0 to 1530
Data columns (total 43 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ImageFile         1531 non-null   object 
 1   RH_x              1531 non-null   float64
 2   WS_x              1531 non-null   float64
 3   AT_x              1531 non-null   float64
 4   PM2.5_x           1531 non-null   float64
 5   SSIM_ROI_x        1531 non-null   float64
 6   ERODE_SSIM_ROI_x  1531 non-null   float64
 7   O_SSIM_ROI_x      1531 non-null   float64
 8   Day_x             1531 non-null   int64  
 9   Month_x           1531 non-null   object 
 10  Hour_x            1531 non-null   int64  
 11  Min_x             1531 non-null   int64  
 12  Month_num_x       1531 non-null   int64  
 13  Day_sin_x         1531 non-null   float64
 14  Day_cos_x         1531 non-null   float64
 15  Mon_sin_x         1531 non-null   float64
 16  Mon_cos_x         1531 non-null   float64


In [68]:
#CALCULATE RESIDUAL
train_belur_df['Res'] = train_belur_df['PM2.5_x'] - train_belur_df['ETR_ROI_y']
test_belur_df['Res'] = test_belur_df['PM2.5_x'] - test_belur_df['ETR_ROI_y']

KeyError: 'ETR_ROI_y'

In [69]:
train_belur_df.columns

Index(['ImageFile', 'RH_x', 'WS_x', 'AT_x', 'PM2.5_x', 'SSIM_ROI_x',
       'ERODE_SSIM_ROI_x', 'O_SSIM_ROI_x', 'Day_x', 'Month_x', 'Hour_x',
       'Min_x', 'Month_num_x', 'Day_sin_x', 'Day_cos_x', 'Mon_sin_x',
       'Mon_cos_x', 'Hour_sin_x', 'Hour_cos_x', 'Min_sin_x', 'Min_cos_x',
       'PM2.5_Norm_x', 'RH_y', 'WS_y', 'AT_y', 'PM2.5_y', 'SSIM_ROI_y',
       'ERODE_SSIM_ROI_y', 'O_SSIM_ROI_y', 'Day_y', 'Month_y', 'Hour_y',
       'Min_y', 'Month_num_y', 'Day_sin_y', 'Day_cos_y', 'Mon_sin_y',
       'Mon_cos_y', 'Hour_sin_y', 'Hour_cos_y', 'Min_sin_y', 'Min_cos_y',
       'PM2.5_Norm_y'],
      dtype='object')

In [70]:
train_b.columns


Index(['ImageFile', 'RH', 'WS', 'AT', 'PM2.5', 'SSIM_ROI', 'ERODE_SSIM_ROI',
       'O_SSIM_ROI', 'Day', 'Month', 'Hour', 'Min', 'Month_num', 'Day_sin',
       'Day_cos', 'Mon_sin', 'Mon_cos', 'Hour_sin', 'Hour_cos', 'Min_sin',
       'Min_cos', 'PM2.5_Norm'],
      dtype='object')

In [71]:
train_bf.columns

Index(['ImageFile', 'RH', 'WS', 'AT', 'PM2.5', 'SSIM_ROI', 'ERODE_SSIM_ROI',
       'O_SSIM_ROI', 'Day', 'Month', 'Hour', 'Min', 'Month_num', 'Day_sin',
       'Day_cos', 'Mon_sin', 'Mon_cos', 'Hour_sin', 'Hour_cos', 'Min_sin',
       'Min_cos', 'PM2.5_Norm'],
      dtype='object')

In [72]:
train_b=pd.read_csv("belur_train.csv")
test_b=pd.read_csv("belur_test.csv")

In [73]:
train_b.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1892 entries, 0 to 1891
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ImageFile       1892 non-null   object 
 1   RH              1892 non-null   float64
 2   WS              1892 non-null   float64
 3   AT              1892 non-null   float64
 4   PM2.5           1892 non-null   float64
 5   SSIM_ROI        1892 non-null   float64
 6   ERODE_SSIM_ROI  1892 non-null   float64
 7   O_SSIM_ROI      1892 non-null   float64
 8   Day             1892 non-null   int64  
 9   Month           1892 non-null   object 
 10  Hour            1892 non-null   int64  
 11  Min             1892 non-null   int64  
 12  Month_num       1892 non-null   int64  
 13  Day_sin         1892 non-null   float64
 14  Day_cos         1892 non-null   float64
 15  Mon_sin         1892 non-null   float64
 16  Mon_cos         1892 non-null   float64
 17  Hour_sin        1892 non-null   f

In [74]:
train_bf=pd.read_csv("belur_frb_train.csv")
test_bf=pd.read_csv("belur_frb_test.csv")

In [75]:
test_bf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 472 entries, 0 to 471
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ImageFile       472 non-null    object 
 1   RH              472 non-null    float64
 2   WS              472 non-null    float64
 3   AT              472 non-null    float64
 4   PM2.5           472 non-null    float64
 5   SSIM_ROI        472 non-null    float64
 6   ERODE_SSIM_ROI  472 non-null    float64
 7   O_SSIM_ROI      472 non-null    float64
 8   Day             472 non-null    int64  
 9   Month           472 non-null    object 
 10  Hour            472 non-null    int64  
 11  Min             472 non-null    int64  
 12  Month_num       472 non-null    int64  
 13  Day_sin         472 non-null    float64
 14  Day_cos         472 non-null    float64
 15  Mon_sin         472 non-null    float64
 16  Mon_cos         472 non-null    float64
 17  Hour_sin        472 non-null    flo

In [76]:
train_belur_df=pd.merge(train_b,train_bf,on='ImageFile',how='inner')
test_belur_df=pd.merge(test_b,test_bf,on='ImageFile',how='inner')

In [77]:
train_belur_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1531 entries, 0 to 1530
Data columns (total 45 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ImageFile         1531 non-null   object 
 1   RH_x              1531 non-null   float64
 2   WS_x              1531 non-null   float64
 3   AT_x              1531 non-null   float64
 4   PM2.5_x           1531 non-null   float64
 5   SSIM_ROI_x        1531 non-null   float64
 6   ERODE_SSIM_ROI_x  1531 non-null   float64
 7   O_SSIM_ROI_x      1531 non-null   float64
 8   Day_x             1531 non-null   int64  
 9   Month_x           1531 non-null   object 
 10  Hour_x            1531 non-null   int64  
 11  Min_x             1531 non-null   int64  
 12  Month_num_x       1531 non-null   int64  
 13  Day_sin_x         1531 non-null   float64
 14  Day_cos_x         1531 non-null   float64
 15  Mon_sin_x         1531 non-null   float64
 16  Mon_cos_x         1531 non-null   float64


In [78]:
test_belur_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 45 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ImageFile         139 non-null    object 
 1   RH_x              139 non-null    float64
 2   WS_x              139 non-null    float64
 3   AT_x              139 non-null    float64
 4   PM2.5_x           139 non-null    float64
 5   SSIM_ROI_x        139 non-null    float64
 6   ERODE_SSIM_ROI_x  139 non-null    float64
 7   O_SSIM_ROI_x      139 non-null    float64
 8   Day_x             139 non-null    int64  
 9   Month_x           139 non-null    object 
 10  Hour_x            139 non-null    int64  
 11  Min_x             139 non-null    int64  
 12  Month_num_x       139 non-null    int64  
 13  Day_sin_x         139 non-null    float64
 14  Day_cos_x         139 non-null    float64
 15  Mon_sin_x         139 non-null    float64
 16  Mon_cos_x         139 non-null    float64
 1

In [79]:
#CALCULATE RESIDUAL
train_belur_df['Res'] = train_belur_df['PM2.5_x'] - train_belur_df['ETR_ROI_y']
test_belur_df['Res'] = test_belur_df['PM2.5_x'] - test_belur_df['ETR_ROI_y']

In [80]:
#NORMALIZE RESIDUAL
train_belur_df['Res_Norm'] = (train_belur_df['Res'] - train_belur_df['Res'].min())/(train_belur_df['Res'].max() - train_belur_df['Res'].min())
test_belur_df['Res_Norm'] = (test_belur_df['Res'] - train_belur_df['Res'].min())/(train_belur_df['Res'].max() - train_belur_df['Res'].min())

In [81]:
#CREATE X LIST FOR BELUR X BELUR RBU DATA
X_train_res = train_belur_df[['SSIM_ROI_x','ERODE_SSIM_ROI_x','Day_sin_x','Day_cos_x','Mon_sin_x','Mon_cos_x','Hour_sin_x','Hour_cos_x','PM2.5_Norm_y']]
X_test_res = test_belur_df[['SSIM_ROI_x','ERODE_SSIM_ROI_x','Day_sin_x','Day_cos_x','Mon_sin_x','Mon_cos_x','Hour_sin_x','Hour_cos_x','PM2.5_Norm_y']]

In [82]:
#CREATE Y LIST FOR BELUR X BELUR RBU DATA
Y_train_res = train_belur_df['Res_Norm']
Y_test_res = test_belur_df['Res_Norm']

In [83]:
#GRID SEARCH
grid = GridSearchCV(ExtraTreesRegressor(),param_grid=param_grid,refit=True,verbose=3,n_jobs=-1)

In [84]:
grid.fit(X_train_res,Y_train_res)

Fitting 5 folds for each of 1616 candidates, totalling 8080 fits
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=20;, score=0.893 total time=   0.0s
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=40;, score=0.904 total time=   0.0s
[CV 4/5] END bootstrap=True, max_features=sqrt, n_estimators=50;, score=0.911 total time=   0.1s
[CV 1/5] END bootstrap=True, max_features=sqrt, n_estimators=70;, score=0.903 total time=   0.1s
[CV 2/5] END bootstrap=True, max_features=sqrt, n_estimators=80;, score=0.903 total time=   0.1s
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=100;, score=0.908 total time=   0.1s
[CV 5/5] END bootstrap=True, max_features=sqrt, n_estimators=110;, score=0.916 total time=   0.1s
[CV 2/5] END bootstrap=True, max_features=sqrt, n_estimators=140;, score=0.908 total time=   0.1s
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=140;, score=0.906 total time=   0.2s
[CV 4/5] END bootstrap=True, max_features=sqrt, n_estimato

/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



[CV 1/5] END bootstrap=True, max_features=0.1, n_estimators=750;, score=0.856 total time=   0.8s
[CV 4/5] END bootstrap=True, max_features=0.1, n_estimators=760;, score=0.874 total time=   1.0s
[CV 2/5] END bootstrap=True, max_features=0.1, n_estimators=780;, score=0.865 total time=   1.0s
[CV 5/5] END bootstrap=True, max_features=0.1, n_estimators=790;, score=0.881 total time=   1.1s
[CV 3/5] END bootstrap=True, max_features=0.1, n_estimators=810;, score=0.874 total time=   0.9s
[CV 1/5] END bootstrap=True, max_features=0.1, n_estimators=830;, score=0.857 total time=   1.0s
[CV 4/5] END bootstrap=True, max_features=0.1, n_estimators=840;, score=0.874 total time=   1.0s
[CV 2/5] END bootstrap=True, max_features=0.1, n_estimators=860;, score=0.863 total time=   1.0s
[CV 5/5] END bootstrap=True, max_features=0.1, n_estimators=870;, score=0.882 total time=   1.0s
[CV 3/5] END bootstrap=True, max_features=0.1, n_estimators=890;, score=0.876 total time=   1.0s
[CV 5/5] END bootstrap=True, 

,estimator,ExtraTreesRegressor()
,param_grid,"{'bootstrap': [True, False], 'max_features': ['sqrt', 'log2', ...], 'n_estimators': array([ 10, ... 1000, 1010])}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,np.int64(490)


In [85]:
grid.best_params_

{'bootstrap': False, 'max_features': 0.7, 'n_estimators': np.int64(490)}

In [86]:
grid.score(X_train_res,Y_train_res)

1.0

In [87]:
grid.score(X_test_res,Y_test_res)

0.9005329666520262

In [88]:
#EXTRA TREE REGRESSOR
etr_belur_res = ExtraTreesRegressor(bootstrap = False, max_features = 0.7, n_estimators = 490, random_state=42)

In [89]:
etr_belur_res.fit(X_train_res,Y_train_res)

,n_estimators,490
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,0.7
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [90]:
etr_belur_res.score(X_train_res,Y_train_res)

1.0

In [91]:
etr_belur_res.score(X_test_res,Y_test_res)

0.9010864012030715


[CV 3/5] END bootstrap=False, max_features=0.5, n_estimators=470;, score=0.917 total time=   0.8s
[CV 1/5] END bootstrap=False, max_features=0.5, n_estimators=490;, score=0.927 total time=   0.8s
[CV 4/5] END bootstrap=False, max_features=0.5, n_estimators=500;, score=0.932 total time=   0.8s
[CV 2/5] END bootstrap=False, max_features=0.5, n_estimators=520;, score=0.917 total time=   0.8s
[CV 5/5] END bootstrap=False, max_features=0.5, n_estimators=530;, score=0.930 total time=   0.9s
[CV 3/5] END bootstrap=False, max_features=0.5, n_estimators=550;, score=0.917 total time=   0.9s
[CV 1/5] END bootstrap=False, max_features=0.5, n_estimators=570;, score=0.926 total time=   0.9s
[CV 4/5] END bootstrap=False, max_features=0.5, n_estimators=580;, score=0.932 total time=   0.9s
[CV 2/5] END bootstrap=False, max_features=0.5, n_estimators=600;, score=0.917 total time=   1.0s
[CV 5/5] END bootstrap=False, max_features=0.5, n_estimators=610;, score=0.930 total time=   1.0s
[CV 3/5] END bootst

In [92]:
#SAVE AS PICKLE FILES
jb.dump(etr_belur_res, 'etr_belur_res.pkl')

['etr_belur_res.pkl']

In [93]:
#PREDICT RESIDUALS
predtrain=etr_belur_res.predict(X_train_res)
predtest=etr_belur_res.predict(X_test_res)

In [94]:
train_belur_df['ETR_Res'] = predtrain * (train_belur_df['Res'].max() - train_belur_df['Res'].min()) + train_belur_df['Res'].min()
test_belur_df['ETR_Res'] = predtest * (train_belur_df['Res'].max() - train_belur_df['Res'].min()) + train_belur_df['Res'].min()

In [95]:
#CALCULATE FINAL PREDICTED PM2.5
train_belur_df['Pred_PM2.5'] = train_belur_df['ETR_ROI_y'] + train_belur_df['ETR_Res']
test_belur_df['Pred_PM2.5'] = test_belur_df['ETR_ROI_y'] + test_belur_df['ETR_Res']


In [96]:
#CHECKING R^2
r2_score(train_belur_df['PM2.5_x'],train_belur_df['Pred_PM2.5'])

1.0

In [97]:
r2_score(test_belur_df['PM2.5_x'],test_belur_df['Pred_PM2.5'])

0.9229865538427776

In [98]:
#SAVE AS CSV
train_belur_df.to_csv('final_train_belur.csv',index=False)
test_belur_df.to_csv('final_test_belur.csv',index=False)


In [1]:
#belurXBELUR

import matplotlib.pyplot as plt
import numpy as np

def scatter_plot(actual, predicted, title):
    plt.figure(figsize=(5,5))

    plt.scatter(
        actual,
        predicted,
        color='red',
        s=10
    )


    mn = min(actual.min(), predicted.min())
    mx = max(actual.max(), predicted.max())

    plt.plot(
        [mn, mx],
        [mn, mx],
        color='green',
        linewidth=1
    )

    plt.xlabel("Measured PM2.5")
    plt.ylabel("Estimated PM2.5")
    plt.title(title)

    plt.tight_layout()
    plt.show()

In [2]:
scatter_plot(
    test_b["PM2.5"],
    test_b["ETR_ROI"],
    "Belur ETR"
)

NameError: name 'test_b' is not defined